# Background

Author: Diane Menuz  
Date: 2026-02-04
Station: Bluff

Goal: Create the qc version of the data


# Set Up

## Parameters

In [ ]:
station = "US-UTJ"
interval_minutes = 30

date_range = '202406051200_202510091200'
parquet_path = r'M:\Shared drives\UGS_Flux\Data_Processing\final_database_tables' 
file_name = f'{station}_{date_range}_raw.parquet'
micromet_path = r"C:/Users/dmenuz/Documents/scripts/MicroMet/src/"

## Libraries

In [ ]:
import sys

sys.path.append(micromet_path)
import micromet
from micromet import validate
from micromet import fix_g_values
from micromet import data_cleaning
from micromet import eddy_plots as ed_plot
from micromet import recalculate_albedo


In [ ]:
sys.path.append(r"C:/Users/dmenuz/Documents/scripts/soil_heat/src/")
from soil_heat import johansen


In [ ]:
import pandas as pd
import numpy as np
import importlib

from prettytable import PrettyTable
from typing import Union

from scipy import stats
import matplotlib.pyplot as plt


from typing import List, Dict, Union

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from plotly.offline import iplot

from pathlib import Path
from datetime import date


In [ ]:
import logging
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
ch = logging.StreamHandler()
ch.setFormatter(
    logging.Formatter(
        fmt="%(levelname)s [%(asctime)s] %(name)s – %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )
)
logger.addHandler(ch)

## Database Data

In [ ]:
import requests
headdict = {'Accept-Profile': 'groundwater',  
            'Content-Type': 'application/json'}  

table = 'eddy_events'
resp = requests.get(f"https://ugs-koop-umfdxaxiyq-wm.a.run.app/{table}",  
                    headers=headdict)
event_db = pd.DataFrame(resp.json())
event_db['event_date'] = pd.to_datetime(event_db['event_date'])

table = 'eddy_station_metadata'
resp = requests.get(f"https://ugs-koop-umfdxaxiyq-wm.a.run.app/{table}",  
                    headers=headdict)
metadata = pd.DataFrame(resp.json())

In [ ]:
notes = event_db.loc[(event_db.stationid==station) & (event_db.event_type=='station visit'),
                     ['event_date','event_notes']]

table = PrettyTable()
table.field_names = ["Date", "Note"]
table.sortby = "Date"
table.align["Note"] = "l"

rows_as_tuples = [tuple(x) for x in notes.values] 
table.add_rows(rows_as_tuples) # Use the correct data format

print(table)

In [ ]:
# review program updates
updates = event_db.loc[(event_db.stationid==station) & 
                       (event_db.event_type=='program update'),
                       ['event_date','event_notes']]

table = PrettyTable()
table.field_names = ["Date", "Note"]
table.sortby = "Date"
table.align["Note"] = "l"

rows_as_tuples = [tuple(x) for x in updates.values] 
table.add_rows(rows_as_tuples) # Use the correct data format

print(table)

## Read in Data

In [ ]:
file_path = Path (parquet_path, 'raw', file_name)
raw_dat = pd.read_parquet(file_path)

# Apply Corrections

- Apply Soil and any other calibraton corrections before running data through the "finalize" function.  
- This ensures that the physical limits are applied to the final values.

## Fix Incorrect SG Calibration

In [ ]:
# date from table above when calibration fix was applied to program
# not sure about this date!!
calibration_fix_date = '2025-11-09 16:30' 

In [ ]:
sg_correct = fix_g_values.correct_vars_by_factor(raw_dat, correction_factor=0.05/0.16, 
                           vars_to_correct=['SG_1_1_1','SG_2_1_1'], 
                           min_correction_date='2010-01-01',
                           max_correction_date=calibration_fix_date)
sg_correct = fix_g_values.calculate_new_g_value(sg_correct, '1')
sg_correct = fix_g_values.calculate_new_g_value(sg_correct, '2')

## Fix Precip Calibration

In [ ]:
# date that precip value fixed in program; site specific!!
precip_program_date = pd.to_datetime('2025-12-01 14:49:00') 
correction_factor = 2.54 # this is likely/hopefully the smae for all sites

In [ ]:
precip_fixed = sg_correct.copy() # copying is key- otherwise if you run this twice, you keep changing the values!
mask = precip_fixed.index<precip_program_date
precip_fixed.loc[mask, 'P_1_1_1'] = precip_fixed.loc[mask, 'P_1_1_1']*correction_factor

In [ ]:
# ed_plot.plotlystuff([precip_fixed, sg_correct],
#                     ['P_1_1_1', 'P_1_1_1'])

## Rename G; Calculate G Surface

In [ ]:
g_ameriflux = precip_fixed.copy()

g_ameriflux['G_1_1_1'] = g_ameriflux['G_PLATE_1_1_1']
g_ameriflux['G_2_1_1'] = g_ameriflux['G_PLATE_2_1_1']
g_ameriflux.drop(columns=['G_PLATE_1_1_1', 'G_PLATE_2_1_1'], inplace=True)
g_ameriflux['G_SURFACE_1_1_1'] = g_ameriflux['G_1_1_1'] + g_ameriflux['SG_1_1_1']
g_ameriflux['G_SURFACE_2_1_1'] = g_ameriflux['G_2_1_1'] + g_ameriflux['SG_2_1_1']

mask = (g_ameriflux['G_1_1_1'].isna()) | (g_ameriflux['SG_1_1_1'].isna())
g_ameriflux.loc[mask, 'G_SURFACE_1_1_1'] = np.nan
mask = (g_ameriflux['G_2_1_1'].isna()) | (g_ameriflux['SG_2_1_1'].isna())
g_ameriflux.loc[mask, 'G_SURFACE_2_1_1'] = np.nan

# Calculate SoilVue G

- Do this before applying physical limits
- SWC values need to be proportion and not percent! They are converted to percent in the Finalize step

In [ ]:
soilvue = g_ameriflux.copy()

In [ ]:
# verify that SWC values are proportions
# will get an assertion error if not (which may just indicate data issues, but 
# inspect data more carefully)
min_swc = min(soilvue['SWC_3_1_1'].min(), soilvue['SWC_3_2_1'].min())
max_swc = max(soilvue['SWC_3_1_1'].max(), soilvue['SWC_3_2_1'].max())
print(f'SWC values between {min_swc} and {max_swc}')
assert min_swc >= 0 and max_swc <= 1, "SWC values are not proportions"


In [ ]:
dt = interval_minutes*60 # calculate the number of seconds in the time interval
DEPTHS_CM = np.array([5, 10, 20, 30, 40, 50, 60], dtype=float)
DEPTHS_M = DEPTHS_CM / 100.0
suffixes = ["3_1_1", "3_2_1", "3_3_1", "3_4_1", "3_5_1", "3_6_1", "3_7_1"]
ts_cols = [f"TS_{s}" for s in suffixes]
swc_cols = [f"SWC_{s}" for s in suffixes]

ts = soilvue[ts_cols].copy()
ts.columns = DEPTHS_CM
swc = soilvue[swc_cols].copy()
swc.columns = DEPTHS_CM

new_values = johansen.gradient_plus_storage(ts, swc, ref_depth_idx=2, dt=dt,lam_model="johansen")
new_values.rename(columns={'G_ref': 'G_3_1_1', 
                           'G_surface': 'G_SURFACE_3_1_1', 
                           'Storage': 'SG_3_1_1'}, inplace=True)
soilvue = soilvue.merge(new_values, left_index=True, right_index=True)

In [ ]:
mask = (soilvue.G_SURFACE_1_1_1<300) & (soilvue.G_SURFACE_1_1_1>-300)
mask2 = (soilvue.G_SURFACE_2_1_1<300) & (soilvue.G_SURFACE_2_1_1>-300)
mask3 = (soilvue.G_SURFACE_3_1_1<300) & (soilvue.G_SURFACE_3_1_1>-300)
subset1 = soilvue[mask & mask2 & mask3]

ed_plot.plotlystuff([subset1, subset1, subset1], 
                    ['G_SURFACE_1_1_1','G_SURFACE_2_1_1', 'G_SURFACE_3_1_1'])

# Micromet Finalize

- This applies physical limits and runs some other corrections

In [ ]:
reformatter = micromet.Reformatter(
    drop_soil=False, logger=logger,)
finalized_df, report, timestamp_results = reformatter.finalize(soilvue)
finalized_df = finalized_df.replace(-9999, np.nan)

In [ ]:
# review and export report
report_name = file_name.replace('raw.parquet', 'report.csv')
report_path = Path(parquet_path, 'micromet_reports', report_name)
report.to_csv(report_path)
print(f'Report exported to {report_path}')
print('\n')
print('Fields with a lot of missing data')
print(report[report.pct_flagged>=0.5])

In [ ]:
validate.validate_flags(finalized_df)

# Address Common Data Issues

In [ ]:
cleaned_df = finalized_df.copy()

## Drop Precip on Field Days

In [ ]:
# review rain on station visits
mask = (event_db.stationid==f'{station}') & (event_db.event_type=='station visit') 
visits = event_db[mask]
print('Precip events that occurred on station visits day and could be dropped')
visit_precip = cleaned_df.loc[(cleaned_df.index.floor('D').isin(
    visits.event_date.dt.floor('D'))) & (cleaned_df.P_1_1_1>0),'P_1_1_1']
visit_precip

In [ ]:
# drop rain on station visit day
cleaned_df.loc[cleaned_df.index.isin(visit_precip.index), 'P_1_1_1'] = 0

## Drop Zeros in G Plates

In [ ]:
mask1 = cleaned_df['G_1_1_1']==0
mask2 = cleaned_df['G_2_1_1']==0
print(f'{mask1.sum()} zero G plate 1 values and {mask2.sum()} zero G plate 2 values')

cleaned_df.loc[mask1, ['G_1_1_1','G_SURFACE_1_1_1']] = np.nan
cleaned_df.loc[mask2, ['G_2_1_1','G_SURFACE_2_1_1']] = np.nan

# Address Other Data Issues

In [ ]:
final_dat = cleaned_df.copy()

## NETRAD

In [ ]:
## NETRAD

# spike in lw_in data to drop
final_dat.loc[final_dat.index==pd.to_datetime('2024-08-30 0:00'), [
    'LW_IN_1_1_1', 'NETRAD_1_1_1']] = np.nan

In [ ]:
# SET ALB to zero if component values are zero
mask = ((final_dat.SW_IN_1_1_1.isna()) | (final_dat.SW_OUT_1_1_1.isna())) & (~final_dat.ALB_1_1_1.isna())
final_dat.loc[mask, ['SW_IN_1_1_1', 'SW_OUT_1_1_1', 'ALB_1_1_1']] = np.nan

mask = ((final_dat.SW_IN_1_1_2.isna()) | (final_dat.SW_OUT_1_1_2.isna())) & (~final_dat.ALB_1_1_2.isna())
final_dat.loc[mask, ['SW_IN_1_1_2', 'SW_OUT_1_1_2', 'ALB_1_1_2']] = np.nan

## Soil

In [ ]:
# ## G VALUES

# # # dropping data with big spike in G_PLATE values, turning all associated data to NA
# # # note that the G values were mostly being dropped anyway
drop1 = pd.to_datetime('2024-08-29 20:00')
drop2 = pd.to_datetime('2024-08-30 0:00')
gspike_mask = (final_dat.index>=drop1) & (final_dat.index<=drop2)
final_dat.loc[gspike_mask, ['G_SURFACE_1_1_1', 'G_SURFACE_2_1_1', 'G_1_1_1','G_2_1_1']] = np.nan

In [ ]:
# SOILVUE

drop_value = 0.30
bad_soilvue_mask = (final_dat.EC_3_7_1<drop_value)
bad_data = final_dat[bad_soilvue_mask]

columns_for_drop = final_dat.columns[final_dat.columns.str.contains('EC_3')].append(
    final_dat.columns[final_dat.columns.str.contains('K_3')]).append(
        final_dat.columns[final_dat.columns.str.contains('SWC_3')]).append(
            final_dat.columns[final_dat.columns.str.contains('TS_3')])

drop_indices = bad_data.index
additional_bad = pd.to_datetime(['2024-10-08 15:30'])
drop_indices = drop_indices.append(additional_bad)
print(f'Number of soilvue values to drop: {len(drop_indices)}')


final_dat.loc[final_dat.index.isin(drop_indices), columns_for_drop] = np.nan

## Misc. Fixes

In [ ]:
## WIND DIRECTION

# adjust wind before that change data by median difference value
# using median difference in wind direction
change_date = pd.to_datetime('2025-07-15 12:30') # date that irgason moved
mask = final_dat.index<change_date

wind_diff = 82

final_dat.loc[mask, 'WD_1_1_1'] = (final_dat.WD_1_1_1-wind_diff)% 360


In [ ]:
## TEMPERATURE

# drop temp spikes
final_dat = data_cleaning.set_range_to_nan(final_dat, 'TA_1_4_1', start_date = '2024-08-29 20:00',end_date='2024-08-30 0:00')
final_dat = data_cleaning.set_range_to_nan(final_dat, 'T_SONIC_1_1_1', start_date = '2024-08-06 10:30',end_date='2024-08-07 6:30')
final_dat = data_cleaning.set_range_to_nan(final_dat, 'T_SONIC_1_1_1', start_date = '2025-06-14 10:30',end_date='2025-06-15 7:30')
final_dat = data_cleaning.set_range_to_nan(final_dat, 'H_1_1_1', start_date = '2024-08-06 10:30',end_date='2024-08-07 6:30')
final_dat = data_cleaning.set_range_to_nan(final_dat, 'H_1_1_1', start_date = '2025-06-14 10:30',end_date='2025-06-15 7:30')

In [ ]:
# PRECIP CORRECTION
# bucket was not working before this date
fix_date = pd.to_datetime('2025-02-14 0:00')
final_dat.loc[(final_dat.index<=fix_date), 'P_1_1_1'] = np.nan

In [ ]:
# Atmospheric Pressure
# huge downward spike

drop_date = pd.to_datetime('2024-06-05 17:30')
mask = final_dat.index==drop_date
final_dat.loc[mask, 'PA_1_1_1'] = np.nan

## Flagging

In [ ]:
# initially set all flags with low signal strength to 1
final_dat['H2O_SIG_FLAG_1_1_1'] = 0

low_sig_mask = final_dat.H2O_SIG_STRGTH_MIN_1_1_1<0.8
final_dat.loc[low_sig_mask, ['H2O_SIG_FLAG_1_1_1']] = 1

# setting flags on the worst chunks of data

flag_cols = ['H2O_SIG_FLAG_1_1_1']

# continuous low signal stretches
final_dat = data_cleaning.apply_internal_flags(final_dat, 
                         flag_cols,
                         '2024-07-25 17:00',
                         '2024-08-18',
                         2)

final_dat = data_cleaning.apply_internal_flags(final_dat, flag_cols,
                                    '2025-04-26 23:00', 
                                    '2025-06-06 12:00', 
                                    2)

final_dat = data_cleaning.apply_internal_flags(final_dat, flag_cols,
                                    '2025-07-17 10:00', 
                                    '2025-08-11 12:30', 
                                    2)

In [ ]:
final_dat['CO2_SIG_FLAG_1_1_1'] = 0

low_sig_mask = final_dat.CO2_SIG_STRGTH_MIN_1_1_1<0.8
final_dat.loc[low_sig_mask, ['CO2_SIG_FLAG_1_1_1']] = 1

# setting flags on the worst chunks of data

flag_cols = ['CO2_SIG_FLAG_1_1_1']

# continuous low signal stretches
final_dat = data_cleaning.apply_internal_flags(final_dat, 
                         flag_cols,
                         '2024-07-25 17:00',
                         '2024-08-18',
                         2)

final_dat = data_cleaning.apply_internal_flags(final_dat, flag_cols,
                                    '2025-04-24 23:00', 
                                    '2025-06-06 12:00', 
                                    2)

final_dat = data_cleaning.apply_internal_flags(final_dat, flag_cols,
                                    '2025-07-17 10:00', 
                                    '2025-08-11 12:30', 
                                    2)

In [ ]:
# set wind flags
# between 0 and 180 is the worst
wind_var = 'WD_1_1_1'
wind_flag = wind_var+'_FLAG'

final_dat[wind_flag] = 0

# set flag for all values between 0 and 180 as 2, really bad
bad_wind_flag2 = (final_dat[wind_var]>=0) & (final_dat[wind_var]<180)
final_dat.loc[bad_wind_flag2, wind_flag] = 2

# set flag for all values between 180 and 190 and 350 and 360 as flag 1
kinda_bad1 = final_dat[wind_var].between(180, 190, inclusive='right')
kinda_bad2 = final_dat[wind_var].between(350, 360, inclusive='right')
kinda_bad_flags = kinda_bad1 | kinda_bad2
final_dat.loc[kinda_bad_flags, wind_flag] = 1

# Create G_1

In [ ]:
ed_plot.plotlystuff([final_dat, final_dat, final_dat], 
                    [ 'G_SURFACE_1_1_1', 'G_SURFACE_2_1_1', 'G_SURFACE_3_1_1'], 
                    chrttitle='G')

In [ ]:
# create gap-filled netrad 2 column
final_dat['G_1'] = final_dat[['G_SURFACE_1_1_1', 'G_SURFACE_2_1_1']].mean(axis=1, skipna=False)

model, model_results = data_cleaning.train_linear_regression_model(
    df = final_dat, 
    target_col = 'G_1', 
    predictor_col = 'G_SURFACE_1_1_1')
print(model_results)
final_dat['G_1'] = data_cleaning.impute_missing_values(
    df = final_dat, 
    model = model, 
    target_col = 'G_1', 
    predictor_col = 'G_SURFACE_1_1_1')

model, model_results = data_cleaning.train_linear_regression_model(
    df = final_dat, 
    target_col = 'G_1', 
    predictor_col = 'G_SURFACE_2_1_1')
print(model_results)
final_dat['G_1'] = data_cleaning.impute_missing_values(
    df = final_dat, 
    model = model, 
    target_col = 'G_1', 
    predictor_col = 'G_SURFACE_2_1_1')

model, model_results = data_cleaning.train_linear_regression_model(
    df = final_dat, 
    target_col = 'G_1', 
    predictor_col = 'G_SURFACE_3_1_1')
print(model_results)
final_dat['G_1'] = data_cleaning.impute_missing_values(
    df = final_dat, 
    model = model, 
    target_col = 'G_1', 
    predictor_col = 'G_SURFACE_3_1_1')

ed_plot.plotlystuff([final_dat, final_dat, final_dat, final_dat], 
                    ['G_1', 'G_SURFACE_1_1_1', 'G_SURFACE_2_1_1', 'G_SURFACE_3_1_1'], 
                    chrttitle='G')

# Export Data

In [ ]:
# add variables for data review
final_dat['day_of_year'] = final_dat.index.dayofyear
final_dat['time_of_day'] = final_dat.index.hour + final_dat.index.minute/60
final_dat['days_since_20240101'] = (final_dat.index - pd.Timestamp('2024-01-01')).days

In [ ]:
file_name = f"{station}_{date_range}_qc.parquet"
file_path = Path(f'{parquet_path}', "qc", f'{file_name}')
final_dat.to_parquet(file_path)
from datetime import datetime
print(f'Data exported {datetime.now()}')
print(f'Exported data to {file_path}')